In [2]:
from __future__ import annotations

import xarray as xr
import os
from pathlib import Path
import re
from typing import Any, Sequence
import pandas as pd
import numpy as np
from datetime import datetime

In [3]:
def read_grib2_timestep(
    grib_paths: Sequence[str | Path],
    *,
    step_hours: int | None = None,
    params: Sequence[str] = ("t2m", "u10", "v10", "tp"),
    data_types: Sequence[str] = ("cf", "pf"),
    member_dim: str = "number",
    filter_by_keys: dict[str, Any] | None = None,
    indexpath: str | None = "",
    engine: str = "cfgrib",
    auto_fix_levels: bool = True,
) -> xr.Dataset:
    """
    Read all GRIB2 files for one timestep and return a single xarray.Dataset.

    - Works when control (cf) is 2D (lat, lon) and perturbations (pf) are 3D
      (number, lat, lon) by adding `number=0` to control fields.
    - If `step_hours` is provided, `grib_paths` may be a mixed list and the function
      selects files whose filename contains '-<step>h-'.
    - Requires: `pip install cfgrib eccodes`
    """
    if engine != "cfgrib":
        raise ValueError("Only engine='cfgrib' is supported")

    if not grib_paths:
        raise ValueError("grib_paths cannot be empty")

    paths = [Path(p) for p in grib_paths]

    if step_hours is not None:
        step_re = re.compile(r"-(\d+)h-", re.IGNORECASE)
        selected: list[Path] = []
        for p in paths:
            m = step_re.search(p.name)
            if m and int(m.group(1)) == int(step_hours):
                selected.append(p)
        if not selected:
            raise KeyError(f"No files matched step_hours={step_hours} using '-<step>h-' in filename")
        paths = selected

    aliases = {"t2m": "2t", "u10": "10u", "v10": "10v", "tp": "tp", "rh2m": "2r", "d2m": "2d"}
    level_recs: dict[str, dict[str, Any]] = {
        "2t": {"typeOfLevel": "heightAboveGround", "level": 2},
        "2r": {"typeOfLevel": "heightAboveGround", "level": 2},
        "2d": {"typeOfLevel": "heightAboveGround", "level": 2},
        "10u": {"typeOfLevel": "heightAboveGround", "level": 10},
        "10v": {"typeOfLevel": "heightAboveGround", "level": 10},
        "tp": {"typeOfLevel": "surface"},
    }
    drop_coords = {
        "heightAboveGround",
        "surface",
        "isobaricInhPa",
        "meanSea",
        "hybrid",
        "depthBelowLandLayer",
    }

    def normalize_param(p: str) -> str:
        p = p.strip().lower()
        if not p:
            raise ValueError("param cannot be empty")
        return aliases.get(p, p)

    def canonicalize(da: xr.DataArray) -> xr.DataArray:
        for c in drop_coords:
            if c in da.dims and da.sizes.get(c, 0) == 1:
                da = da.squeeze(c, drop=True)
            if c in da.coords and c not in da.dims:
                da = da.reset_coords(c, drop=True)
        return da

    def ensure_member_dim(da: xr.DataArray) -> xr.DataArray:
        if member_dim in da.dims:
            return da
        if member_dim in da.coords:
            v = da.coords[member_dim].values
            try:
                v = int(v)
            except Exception:
                v = 0
            da = da.reset_coords(member_dim, drop=True)
            return da.expand_dims({member_dim: [v]})
        return da.expand_dims({member_dim: [0]})

    def open_datasets(path: Path, fbk: dict[str, Any]) -> list[xr.Dataset]:
        import cfgrib  # type: ignore

        try:
            return cfgrib.open_datasets(
                str(path),
                backend_kwargs={"filter_by_keys": fbk, "indexpath": indexpath},
            )
        except Exception:
            return []

    def find_short_name(datasets: list[xr.Dataset], short_name: str) -> xr.DataArray | None:
        for ds in datasets:
            if short_name in ds.data_vars:
                return ds[short_name]
            for _, da in ds.data_vars.items():
                if da.attrs.get("GRIB_shortName") == short_name:
                    return da
        return None

    def concat_members(arrays: list[xr.DataArray]) -> xr.DataArray:
        if len(arrays) == 1:
            return arrays[0]
        combined = xr.concat(arrays, dim=member_dim)
        if member_dim in combined.coords:
            try:
                vals = np.asarray(combined[member_dim].values).reshape(-1)
                _, idx = np.unique(vals, return_index=True)
                if len(idx) != len(vals):
                    combined = combined.isel({member_dim: np.sort(idx)})
            except Exception:
                pass
        return combined.sortby(member_dim)

    base = dict(filter_by_keys or {})
    base.pop("dataType", None)
    out: dict[str, xr.DataArray] = {}

    for display_name in params:
        short_name = normalize_param(display_name)
        arrays: list[xr.DataArray] = []

        for path in paths:
            for dt in data_types:
                fbk = dict(base)
                fbk["dataType"] = str(dt)
                fbk["shortName"] = short_name

                dss = open_datasets(path, fbk)
                da = find_short_name(dss, short_name)

                if da is None and auto_fix_levels and short_name in level_recs:
                    fbk.update(level_recs[short_name])
                    dss = open_datasets(path, fbk)
                    da = find_short_name(dss, short_name)

                if da is None:
                    continue

                da = ensure_member_dim(canonicalize(da))
                arrays.append(da)

        if not arrays:
            raise KeyError(f"Param {display_name!r} (shortName {short_name!r}) not found in provided files")

        out[str(display_name)] = concat_members(arrays)

    return xr.Dataset(out)

In [4]:
def wrap_lon_180(lon): return ((lon + 180) % 360) - 180

def select_cell_containing_tp(ds_tp: xr.DataArray, lats, lons):
    lat_vals = ds_tp["latitude"].values
    lon_vals = ds_tp["longitude"].values
    dlat = float(np.abs(lat_vals[1] - lat_vals[0]))
    dlon = float(np.abs(lon_vals[1] - lon_vals[0]))
    half_dlat = dlat / 2.0
    half_dlon = dlon / 2.0

    lat_desc = lat_vals[0] > lat_vals[-1]
    if lat_desc:
        lat_asc = lat_vals[::-1]
        lat_edges = np.concatenate([
            [lat_asc[0] - half_dlat],
            (lat_asc[:-1] + lat_asc[1:]) / 2.0,
            [lat_asc[-1] + half_dlat],
        ])
        lat_bin = np.searchsorted(lat_edges, lats, side="right") - 1
        lat_bin = np.clip(lat_bin, 0, len(lat_asc) - 1)
        lat_idx = (len(lat_vals) - 1) - lat_bin
    else:
        lat_edges = np.concatenate([
            [lat_vals[0] - half_dlat],
            (lat_vals[:-1] + lat_vals[1:]) / 2.0,
            [lat_vals[-1] + half_dlat],
        ])
        lat_idx = np.searchsorted(lat_edges, lats, side="right") - 1
        lat_idx = np.clip(lat_idx, 0, len(lat_vals) - 1)

    lon_edges = np.concatenate([
        [lon_vals[0] - half_dlon],
        (lon_vals[:-1] + lon_vals[1:]) / 2.0,
        [lon_vals[-1] + half_dlon],
    ])
    lon_idx = np.searchsorted(lon_edges, lons, side="right") - 1
    lon_idx = np.clip(lon_idx, 0, len(lon_vals) - 1)

    lat_centers = xr.DataArray(lat_vals[lat_idx], dims="point")
    lon_centers = xr.DataArray(lon_vals[lon_idx], dims="point")

    return ds_tp.sel(latitude=lat_centers, longitude=lon_centers)

def build_station_table(ds: xr.Dataset, coords: pd.DataFrame, filename: str | Path) -> pd.DataFrame:
    lats = coords["latitude"].to_numpy(dtype=float)
    lons = np.array([wrap_lon_180(x) for x in coords["longitude"].to_numpy(dtype=float)])

    lat_da = xr.DataArray(lats, dims="point")
    lon_da = xr.DataArray(lons, dims="point")

    cont = ds[["t2m", "u10", "v10"]].interp(latitude=lat_da, longitude=lon_da, method="linear")

    tp_cc = select_cell_containing_tp(ds["tp"], lats=lats, lons=lons).rename("tp")

    cont2 = cont.reset_coords(names=["latitude", "longitude"], drop=True)
    tp2 = tp_cc.reset_coords(names=["latitude", "longitude"], drop=True)

    combined = xr.merge([cont2, tp2])

    df = combined.to_dataframe().reset_index()  

    df["city_name"] = df["point"].map(dict(enumerate(coords["name"].tolist())))
    df["lat"] = df["point"].map(dict(enumerate(lats.tolist())))
    df["lon"] = df["point"].map(dict(enumerate(lons.tolist())))

    m = re.search(r"(\d{14})", Path(filename).name)
    if not m:
        raise ValueError(f"Could not find YYYYMMDDhhmmss timestamp in filename: {filename!r}")
    df["forecasted_at_utc"] = pd.to_datetime(m.group(1), format="%Y%m%d%H%M%S")

    if "valid_time" in ds:
        dt_utc = pd.to_datetime(ds["valid_time"].values)
    else:
        dt_utc = pd.to_datetime(ds["time"].values + ds["step"].values)

    df["datetime_utc"] = dt_utc
    df[["t2m", "u10", "v10", "tp"]] = df[["t2m", "u10", "v10", "tp"]].round(2)

    return df[["forecasted_at_utc", "datetime_utc", "city_name", "number", "t2m", "u10", "v10", "tp"]]

def _safe_city_dir_name(city_name: str) -> str:
    city_name = str(city_name).strip()
    if not city_name:
        return "unknown"
    return re.sub(r"[<>:\"/\\|?*]", "_", city_name)

def _fmt_yyyymmddhhmm(dt) -> str:
    return pd.Timestamp(dt).strftime("%Y%m%d%H%M")

def write_city_parquets(
    result: pd.DataFrame,
    *,
    base_dir: str | Path = Path("data") / "point_data" / "ecmwf-ensemble",
) -> list[Path]:
    """Write/append per-city parquet files under:
    data/point_data/ecmwf-ensemble/<city_name>/raw/ecmwf-ensemble_indexes_<forecasted_at>_<max_datetime>.parquet
    """
    base_dir = Path(base_dir)
    if result is None or len(result) == 0:
        return []

    required = {"city_name", "forecasted_at_utc", "datetime_utc"}
    missing = required - set(result.columns)
    if missing:
        raise ValueError(f"result is missing required columns: {sorted(missing)}")

    df = result.copy()
    df["forecasted_at_utc"] = pd.to_datetime(df["forecasted_at_utc"])
    df["datetime_utc"] = pd.to_datetime(df["datetime_utc"])

    written: list[Path] = []
    for (city_name, forecasted_at_utc), g in df.groupby(["city_name", "forecasted_at_utc"], dropna=False):
        city_dir = base_dir / _safe_city_dir_name(str(city_name)) / "raw"
        city_dir.mkdir(parents=True, exist_ok=True)

        forecasted_s = _fmt_yyyymmddhhmm(forecasted_at_utc)
        pattern = f"ecmwf-ensemble_indexes_{forecasted_s}_*.parquet"
        existing = list(city_dir.glob(pattern))

        def _end_key(p: Path) -> str:
            m = re.search(rf"ecmwf-ensemble_indexes_{forecasted_s}_(\\d{{12}})\\.parquet$", p.name)
            return m.group(1) if m else ""

        existing_sorted = sorted(existing, key=_end_key)
        existing_df = None
        if existing_sorted:
            try:
                existing_df = pd.read_parquet(existing_sorted[-1])
            except ImportError as e:
                raise ImportError("Parquet read/write requires 'pyarrow'. Install it with: pip install pyarrow") from e

        combined = g if existing_df is None else pd.concat([existing_df, g], ignore_index=True)
        combined["forecasted_at_utc"] = pd.to_datetime(combined["forecasted_at_utc"])
        combined["datetime_utc"] = pd.to_datetime(combined["datetime_utc"])

        if "number" in combined.columns:
            combined["number"] = pd.to_numeric(combined["number"], errors="coerce").astype("Int64")

        dedup_subset = [c for c in ["forecasted_at_utc", "datetime_utc", "number"] if c in combined.columns]
        if dedup_subset:
            combined = combined.drop_duplicates(subset=dedup_subset, keep="last")

        sort_cols = [c for c in ["forecasted_at_utc", "datetime_utc", "number"] if c in combined.columns]
        if sort_cols:
            combined = combined.sort_values(sort_cols, kind="mergesort")

        preferred_cols = [c for c in result.columns if c in combined.columns] + [c for c in combined.columns if c not in result.columns]
        combined = combined[preferred_cols]
        combined = combined.reset_index(drop=True)

        end_dt = combined["datetime_utc"].max()
        end_s = _fmt_yyyymmddhhmm(end_dt)
        out_path = city_dir / f"ecmwf-ensemble_indexes_{forecasted_s}_{end_s}.parquet"

        for p in existing:
            if p.resolve() != out_path.resolve() and p.exists():
                p.unlink()

        try:
            combined.to_parquet(out_path, index=False)
        except ImportError as e:
            raise ImportError("Parquet read/write requires 'pyarrow'. Install it with: pip install pyarrow") from e

        written.append(out_path)

    return written


In [5]:
locations = pd.read_csv("locations.csv")

coords = (
    locations
    .assign(
        latitude=lambda d: (
            d["lat_lon"]
            .str.extract(r'(\d+(?:\.\d+)?)([NS])')[0]
            .astype(float)
            * d["lat_lon"].str.extract(r'(\d+(?:\.\d+)?)([NS])')[1].map({"N": 1, "S": -1})
        ),
        longitude=lambda d: (
            d["lat_lon"]
            .str.extract(r'([0-9.]+)([EW])$', expand=True)[0]
            .astype(float)
            * d["lat_lon"].str.extract(r'([0-9.]+)([EW])$', expand=True)[1].map({"E": 1, "W": -1})
        )
    )
    [["name", "latitude", "longitude"]]
)

In [ ]:
with os.scandir(Path.cwd() / "data" / "raster_data" / "ecmwf-ensemble" / "18z" / "20260105") as entries: files = [e.path for e in entries if e.is_file() and e.name.endswith(".grib2")]
timesteps = sorted({str(re.search(r'-(\d+)h-', f).group(1) + "h") for f in files})
for t in timesteps:
    print(f"Processing timestep: {t} | {datetime.now().strftime('%Y%m%d %H%M%S')}")
    files_of_timestep = [f for f in files if f"-{t}-" in Path(f).name]
    ds = read_grib2_timestep(files_of_timestep)
    result = build_station_table(ds, coords, filename=files_of_timestep[0])
    written_paths = write_city_parquets(result)
    written_paths

In [ ]:
import pyarrow.parquet as pq
atlanta = pq.read_table(r"C:\Users\gkbrk\Desktop\polymarket-weather-trading\data\point_data\ecmwf-ensemble\Atlanta\raw\ecmwf-ensemble_indexes_202601050600_202601200600.parquet").to_pandas()
atlanta


,forecasted_at_utc,datetime_utc,city_name,number,t2m,u10,v10,tp
0,2026-01-05 06:00:00,2026-01-05 06:00:00,Atlanta,0,280.51,-1.21,0.89,0.000000
1,2026-01-05 06:00:00,2026-01-05 06:00:00,Atlanta,1,280.84,-1.42,1.52,0.000000
2,2026-01-05 06:00:00,2026-01-05 06:00:00,Atlanta,2,280.52,-1.56,0.39,0.000000
3,2026-01-05 06:00:00,2026-01-05 06:00:00,Atlanta,3,280.77,-1.29,0.70,0.000000
4,2026-01-05 06:00:00,2026-01-05 06:00:00,Atlanta,4,279.82,-0.64,1.26,0.000000
...,...,...,...,...,...,...,...,...
3106,2026-01-05 06:00:00,2026-01-20 06:00:00,Atlanta,46,267.58,1.75,-1.50,27.920000
3107,2026-01-05 06:00:00,2026-01-20 06:00:00,Atlanta,47,276.30,0.39,-1.53,102.800003
3108,2026-01-05 06:00:00,2026-01-20 06:00:00,Atlanta,48,283.74,1.43,1.95,21.590000
3109,2026-01-05 06:00:00,2026-01-20 06:00:00,Atlanta,49,285.87,1.29,5.14,17.639999


: 